In [30]:
#IMPORTS
import streamlit as st
from fileinput import filename
import pandas as pd
import hvplot.pandas
from path import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from st_aggrid import GridOptionsBuilder, AgGrid, GridUpdateMode, DataReturnMode
import holoviews as hv
import numpy as np
import sys
import os
from pathlib import Path, WindowsPath, PureWindowsPath
sys.path.append(str(Path.cwd().parent))
from Intelligence import Utils, cloud_data_collector, DataFramer, Helper
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [31]:
## Welcome to Company Predictive Analytics
def GetSelectedTicker():
        ticker = 'NVDA'
        return ticker
    
ticker = GetSelectedTicker()
tickerdetails = None
TotalProfitLoss = 0


In [32]:
abspath = os.path.abspath("")
#print(abspath)
tickerdatapath1 = Path(abspath).parent
#print(tickerdatapath1)
location = tickerdatapath1 / 'TickerData' # type: ignore
#print(location)
dfTicker:pd.DataFrame = Utils.GetLocalTickerData(ticker, location)
#print(f'Loading ticker data for ticker {ticker}')
#print(dfTicker)
if(dfTicker is None):
    dfTicker = cloud_data_collector.GetYFinanceData(ticker, '2y')
    if(dfTicker is not None):
        dfTicker.drop('Adj Close', axis=1, inplace=True)
        Utils.SaveTickerData(ticker, dfTicker, location)

if(dfTicker is None):
    print(f"Error loading data for ticker {ticker}.")             
else:
    print(f'Loading ticker data for ticker {ticker}')
    print(dfTicker.head())
    print(dfTicker.tail())


Loading ticker data for ticker NVDA
                  Open        High         Low       Close    Volume
Date                                                                
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800
                  Open        High         Low       Close    Volume
Date                                                                
2023-09-27  423.299988  428.720001  416.290009  424.679993  44493500
2023-09-28  424.600006  434.459991  421.149994  430.890015  42466300
2023-09-29  438.269989  441.440002  433.070007  434.989990  39722100
2023-10-02  440.299988  451.750000  438.609985  447.820007  43329800
2023-10-03  448.079987  451.299988  432.459991  435.170013  4697070

In [33]:
dfPlot = dfTicker.hvplot.line(
    height=600,
    rot=90,
    responsive=True
)
hv.render(dfPlot, backend='bokeh')

Figure(id='1702', ...)

In [34]:
#Get the Difference between today and previous day
dfTicker['Return'] = dfTicker.Close.diff(periods=1)
dfTicker['Return'].fillna(0, inplace=True)
# Calculate the daily returns using the closing prices and the pct_change function
dfTicker["PercentReturn"] = dfTicker["Close"].pct_change()

print(dfTicker.head())
print(dfTicker.tail())


                  Open        High         Low       Close    Volume    Return  PercentReturn
Date                                                                                         
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500       NaN            NaN
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200  7.189987       0.036438
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200  2.490005       0.012175
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900  3.750000       0.018116
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800 -2.440002      -0.011578
                  Open        High         Low       Close    Volume     Return  PercentReturn
Date                                                                                          
2023-09-27  423.299988  428.720001  416.290009  424.679993  44493500   5.570007       0.013290
2023-09-28  424.600006  434.459991  421.149994  430.89001

In [35]:
dfTicker['DailyTrend'] = dfTicker['Return'].apply(Helper.CalcTrend)
dfTicker['DailyTrend'].fillna(0, inplace=True)
dfTicker.DailyTrend = dfTicker.DailyTrend.astype('int')
dfTicker['Trend'] = 0
print(dfTicker.head())
print(dfTicker.tail())


                  Open        High         Low       Close    Volume    Return  PercentReturn  DailyTrend  Trend
Date                                                                                                            
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500       NaN            NaN         NaN      0
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200  7.189987       0.036438         1.0      0
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200  2.490005       0.012175         1.0      0
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900  3.750000       0.018116         1.0      0
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800 -2.440002      -0.011578        -1.0      0
                  Open        High         Low       Close    Volume     Return  PercentReturn  DailyTrend  Trend
Date                                                                                           

In [36]:
# Using multiple apply arguments
prevdailytrend = 0
prevtrend = 0
dailytrend = 0
trend = 0

for i in range(len(dfTicker)):
   #print(dfTicker.loc[i, "Name"], dfTicker.loc[i, "Age"])
    #previousrow = dfTicker.iloc[[i-1]]
    dailytrend = dfTicker["DailyTrend"].iloc[i]
    if(i==0): #First Row of the DataSet
        if(dailytrend is np.nan or dailytrend==0):
            dfTicker.loc[dfTicker.index[i], "Trend"] = 0
    else:
        prevdailytrend = dfTicker["DailyTrend"].iloc[i-1]
        prevtrend = dfTicker["Trend"].iloc[i-1]
        trend = 0
        if(prevtrend>=0):                       # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = prevtrend + dailytrend  # type:ignore
            elif( dailytrend<0):                # type:ignore
                trend = dailytrend
        elif(prevtrend<0):                      # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = dailytrend
            elif( dailytrend<0):                # type:ignore
                trend = prevtrend + dailytrend # type:ignore
        dfTicker.loc[dfTicker.index[i], "Trend"] = trend
print(dfTicker.head())
print(dfTicker.tail())


                  Open        High         Low       Close    Volume    Return  PercentReturn  DailyTrend  Trend
Date                                                                                                            
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500       NaN            NaN         NaN      0
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200  7.189987       0.036438         1.0      1
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200  2.490005       0.012175         1.0      2
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900  3.750000       0.018116         1.0      3
2021-10-08  211.009995  212.059998  207.750000  208.309998  15125800 -2.440002      -0.011578        -1.0     -1
                  Open        High         Low       Close    Volume     Return  PercentReturn  DailyTrend  Trend
Date                                                                                           

In [37]:


# Set the variables for short window and long window periods
sma9 = 9
sma20 = 20
sma50 = 50
sma100 = 100
sma200 = 200
print('Generate the SMAs')
dfTicker["SMA9"] = dfTicker["Close"].rolling(window=sma9).mean()
dfTicker["SMA20"] = dfTicker["Close"].rolling(window=sma20).mean()
dfTicker["SMA50"] = dfTicker["Close"].rolling(window=sma50).mean()
dfTicker["SMA100"] = dfTicker["Close"].rolling(window=sma100).mean()
dfTicker["SMA200"] = dfTicker["Close"].rolling(window=sma200).mean()

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

Generate the SMAs
                  Open        High         Low       Close    Volume     Return  PercentReturn  DailyTrend  Trend        SMA9       SMA20       SMA50      SMA100     SMA200
Date                                                                                                                                                                        
2021-10-04  205.039993  205.419998  195.550003  197.320007  34563500        NaN            NaN         NaN      0         NaN         NaN         NaN         NaN        NaN
2021-10-05  199.500000  206.479996  198.539993  204.509995  27928200   7.189987       0.036438         1.0      1         NaN         NaN         NaN         NaN        NaN
2021-10-06  201.199997  207.199997  200.800003  207.000000  29720200   2.490005       0.012175         1.0      2         NaN         NaN         NaN         NaN        NaN
2021-10-07  210.919998  213.220001  209.720001  210.750000  25691900   3.750000       0.018116         1.0      3    

In [38]:
lstDecimalCols = ['Open', 'High', 'Low', 'Close', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
dfTicker = DataFramer.SetDataFrameColsTo2Decimals(dfData=dfTicker, lstcols=lstDecimalCols)

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

              Open    High     Low   Close    Volume     Return  PercentReturn  DailyTrend  Trend    SMA9   SMA20   SMA50  SMA100  SMA200
Date                                                                                                                                     
2021-10-04  205.04  205.42  195.55  197.32  34563500        NaN            NaN         NaN      0     nan     nan     nan     nan     nan
2021-10-05  199.50  206.48  198.54  204.51  27928200   7.189987       0.036438         1.0      1     nan     nan     nan     nan     nan
2021-10-06  201.20  207.20  200.80  207.00  29720200   2.490005       0.012175         1.0      2     nan     nan     nan     nan     nan
2021-10-07  210.92  213.22  209.72  210.75  25691900   3.750000       0.018116         1.0      3     nan     nan     nan     nan     nan
2021-10-08  211.01  212.06  207.75  208.31  15125800  -2.440002      -0.011578        -1.0     -1     nan     nan     nan     nan     nan
...            ...     ...     ...